In [1]:
from ultralytics import YOLO
from pathlib import Path
from onnxruntime.quantization import quantize_dynamic, QuantType
import os
import datetime

In [2]:
# Configuration

# Data YAML path
DATA_YAML = "data/data.yaml"

# Base model for fine-tuning
BASE_MODEL = "yolov8n.pt"

# Training settings
EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
DEVICE = "0"  # "0" for first GPU, "cpu" for CPU
PROJECT = "runs/train"
NAME = "defect"

# Resume training if a run with the same name exists
RESUME_IF_EXISTS = True

# Export filenames
ONNX_NAME = "best_defect_detector.onnx"
ONNX_INT8_NAME = "best_defect_detector_int8.onnx"

print("DATA_YAML:", DATA_YAML)
print("BASE_MODEL:", BASE_MODEL)


DATA_YAML: data/data.yaml
BASE_MODEL: yolov8n.pt


In [3]:
# Cell 3: Helpers to find last/best run

def get_latest_run(project: str) -> Path | None:
    # Return the most recently modified run directory inside project, or None
    p = Path(project)
    if not p.exists():
        return None
    runs = [d for d in p.iterdir() if d.is_dir()]
    if not runs:
        return None
    return sorted(runs, key=lambda x: x.stat().st_mtime)[-1]

def find_best_pt(run_dir: Path) -> Path | None:
    # Return path to best.pt inside run_dir/weights if it exists, else None
    cand = run_dir / "weights" / "best.pt"
    return cand if cand.exists() else None


In [4]:
# Training with checkpointing and optional resume

exp_dir = Path(PROJECT) / NAME
latest_run = get_latest_run(PROJECT)

resume_flag = False
if RESUME_IF_EXISTS and latest_run is not None and latest_run.name == NAME:
    print(f"[INFO] Found existing run: {latest_run}, will resume.")
    resume_flag = True
else:
    print("[INFO] Starting new run.")

model = YOLO(BASE_MODEL)

train_args = dict(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=PROJECT,
    name=NAME,
    save=True,
    save_period=1,
    exist_ok=True,
)

if resume_flag:
    results = model.train(resume=True, **train_args)
else:
    results = model.train(**train_args)

print("[INFO] Training finished.")


[INFO] Starting new run.
New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.10.12 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3768MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=defe

In [5]:
# Export best.pt to ONNX

run_dir = get_latest_run(PROJECT)
if run_dir is None:
    raise RuntimeError("No run directory found!")

best_pt = find_best_pt(run_dir)
if best_pt is None:
    raise RuntimeError("best.pt not found in run dir!")

print("[INFO] Using best.pt:", best_pt)

# Load best.pt model
best_model = YOLO(best_pt)

# Target ONNX path
onnx_path = run_dir / "weights" / ONNX_NAME

print("[INFO] Exporting to ONNX:", onnx_path)
# Export to ONNX (Ultralytics will write default best.onnx into weights)
best_model.export(format="onnx", opset=12, dynamic=True)

# If default ONNX filename differs, rename to desired name
default_onnx = run_dir / "weights" / "best.onnx"
if default_onnx.exists() and default_onnx != onnx_path:
    default_onnx.rename(onnx_path)

print("[OK] ONNX saved at:", onnx_path)


[INFO] Using best.pt: runs/train/defect/weights/best.pt
[INFO] Exporting to ONNX: runs/train/defect/weights/best_defect_detector.onnx
Ultralytics 8.3.229 🚀 Python-3.10.12 torch-2.7.1+cu118 CPU (Intel Core i7-10870H 2.20GHz)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs/train/defect/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)

ONNX: starting export with onnx 1.19.1 opset 12...
ONNX: slimming with onnxslim 0.1.74...
ONNX: export success ✅ 2.7s, saved as 'runs/train/defect/weights/best.onnx' (12.1 MB)

Export complete (3.0s)
Results saved to /home/muhammad-ammar/AI/QC-SCM/fine-tuning/combine-fine-tuning/defect-YOLO/runs/train/defect/weights
Predict:         yolo predict task=detect model=runs/train/defect/weights/best.onnx imgsz=640  
Validate:        yolo val task=detect model=runs/train/defect/weights/best.onnx imgsz=640 data=data/data.yaml  
Visualize:       https://net

In [6]:
# Quantize the exported ONNX model to INT8
run_dir = get_latest_run(PROJECT)
weights_dir = run_dir / "weights"
onnx_path = weights_dir / ONNX_NAME
onnx_int8_path = weights_dir / ONNX_INT8_NAME

print("[INFO] Input model :", onnx_path)
print("[INFO] Output model:", onnx_int8_path)

# Perform dynamic quantization
quantize_dynamic(
    model_input=str(onnx_path),
    model_output=str(onnx_int8_path),
    weight_type=QuantType.QUInt8,
)

print("[OK] Quantized model saved to:", onnx_int8_path)


[INFO] Input model : runs/train/defect/weights/best_defect_detector.onnx
[INFO] Output model: runs/train/defect/weights/best_defect_detector_int8.onnx
[OK] Quantized model saved to: runs/train/defect/weights/best_defect_detector_int8.onnx
